# Results of the benchmark runs

The tables and figures of the results, in the order of a paper's results
section. Each one is also written to `evaluation_results/paper/`, tables as
CSV and LaTeX and figures as PDF and PNG.

Make the runs first with one job, `evaluation/minerva/run_pipeline.lsf` on
Minerva or `evaluation/local/run_pipeline.sh` on your own machine
([`evaluation/`](../evaluation/README.md)). Then open the notebook from the
Gentrex folder and choose Run, Run All Cells:

```bash
uv run --group notebook jupyter lab evaluation_results/analysis.ipynb
```

[walkthrough.ipynb](walkthrough.ipynb) shows what the four commands do to
one report.

In [ ]:
# The notebook group provides pandas and matplotlib.
# pyright: reportMissingImports=false

import benchruns as bench

runs = bench.load_runs()
bench.runs_table(runs)

A setup is a copy of the reports (`native`, `scan` or `fax`), with
`no select` for a run without `gentrex select`, and with the label of a
run that was scored with `--label`, as one with another model. The
notebook takes the newest run of each setup.

- `RUN` is the run that the sections about one run look at. By default it
  is the one on the native PDFs.

In [ ]:
chosen = {setup: runs[name] for setup, name in bench.newest(runs).items()}
assert chosen, "No scored run under results/gentrex-bench"
RUN = bench.newest(runs).get("native") or next(iter(chosen))
run, errors = runs[RUN], bench.errors(RUN)

# Where the tables and figures are written. None writes nothing.
bench.EXPORT = bench.ROOT / "evaluation_results" / "paper"
bench.headline(run, chosen)

## 1. Data and setup

**Table 1.** The benchmark reports. A stated value is one that a report
prints and Gentrex should extract.

In [ ]:
bench.data_table(run)

**Table 2.** What produced the numbers.

In [ ]:
bench.setup_table(run)

## 2. The whole pipeline

**Table 3.** The main numbers of each setup, in percent with 95% intervals.

| Number | Meaning |
| --- | --- |
| Report accuracy | Reports with nothing clinically wrong: the result, the karyotype, and every variant with its gene, DNA change, copy number, classification and zygosity, and no extra variant |
| Every value right | Reports whose whole record is right, with identifiers, dates and standard values |
| MRN and date of birth right | Reports that would be filed under the right patient |
| Value F1 (micro) | Over all stated values pooled across reports, how many Gentrex got right (recall) and how many of its values are right (precision) |
| Field F1 (macro) | Unweighted mean F1 across every extracted field, so rare clinical fields weigh the same as common ones |
| Clinical variant F1 | Variants that match on gene and DNA change and also have the right copy number, classification and zygosity |


In [ ]:
bench.main_table(chosen)

**Table 3b.** Clinical safety, grounding, and category macro-F1 across setups.
Negative-report specificity and false-positive variant rate are measured on reports with no stated variants.


In [ ]:
bench.safety_table(chosen)

**Table 3c.** Clinical errors by severity category across setups.

In [ ]:
bench.clinical_errors_table(chosen)

**Figure 1.** Report accuracy of each setup, and its difference to the
native PDFs on the same reports. A difference whose interval crosses 0
can't be told from chance.

In [ ]:
bench.accuracy_figure(chosen)

**Figure 2.** Of 100 stated values, how many each command lost. A value is
charged to the first command that lost it: `gentrex transcribe` if its text
isn't in the transcribed pages, `gentrex select` if it is on a dropped
page, and `gentrex extract` otherwise.

In [ ]:
bench.stages_figure(chosen)

**Table 4.** The numbers of Figure 2, per 100 stated values. The last column
counts the values that a record holds and no report states.

In [ ]:
bench.stages_table(chosen)

## 3. Each command on its own

Each command is measured on what it got from the command before, so a loss
upstream doesn't count against it.

### `gentrex transcribe`

**Table 5.** How well each copy was read. OCR word recall is the share of
the true words of a page that the OCR model wrote, and precision the share
of its words that are on the page. Words are compared whatever their order,
because OCR reads the columns of a page in another order than the PDF's
text layer.

In [ ]:
bench.transcribe_table(chosen)

### `gentrex select`

**Table 6.** The pages that `gentrex select` kept. A needed page is the
only page of its report that prints a stated value, so dropping it loses
that value. A stated value is on a kept page if any page that prints it was
kept.

In [ ]:
bench.select_pages_table(chosen)

**Figure 3.** The same at other values of `select.keep_threshold`, from the
probabilities of "yes" that `RUN` wrote. Up and to the left is better: more
values on a kept page with fewer pages.

In [ ]:
bench.select_figure(run)

**Table 7.** The pages that only one question kept. Without that question
they would be dropped.

In [ ]:
bench.questions_table(run)

**Table 8.** The pipeline with and without `gentrex select`, on the same
copy.

In [ ]:
bench.select_table(chosen)

### `gentrex extract`

**Table 9.** How well `gentrex extract` copied the values that reached it,
leaving out the values that `gentrex transcribe` or `gentrex select` lost.

In [ ]:
bench.extract_table(chosen)

**Figure 4.** F1 of every field of `RUN`. `OTHER` names a second setup to
draw next to it. Table A1 in the appendix has the numbers, with the recall
of each field on the values that reached `gentrex extract`.

In [ ]:
OTHER = "fax"
bench.fields_figure(run, chosen.get(OTHER))

**Figure 5.** What the values that aren't right are, in the fields with
the most of them.

| Kind | Meaning | What helps |
| --- | --- | --- |
| Missed | The record leaves out a value that the report states | A page that `gentrex select` keeps, or a clearer field |
| Part of the value | The record holds part of the true value, or more than it | A clearer description of where the value ends |
| Another value of the report | The record holds a value that the report prints for another field or purpose | A sharper description of the field |
| Not in the pages | The record holds text that the pages don't have | A model that copies instead of writing |
| Other text of the pages | The record holds other text of the pages | A sharper description of the field |
| Wrong category or number | A category or a number that isn't right | The field's list of categories |

In [ ]:
bench.errors_figure(run)

**Table 10.** Whole variants and clinical features. An extra one is a
distractor if the report prints it for someone else, as a variant from a
cited study, and made up otherwise.

In [ ]:
bench.entities_table(run)

**Table 11.** Each kind of variant. The accuracy of the classification, the
zygosity and the copy number counts the variants of the kind that were
found.

In [ ]:
bench.variant_kinds_table(run)

**Figure 6.** The classifications of the found variants and the result
categories of the reports, as the report states them (rows) and as Gentrex
wrote them (columns). The shade is the share of the row.

In [ ]:
bench.confusion_figure(run)

### `gentrex normalize`

**Table 12.** The standard values. `End to end` is what a user of the
record gets. `Given a right source` leaves out the values that were
already extracted wrong, so it measures `gentrex normalize` alone.

In [ ]:
bench.normalize_table(run)

**Table 13.** The HPO terms of the clinical features whose text was found.

The benchmark prints each feature as an HPO label or synonym, and each
condition as its MONDO label, so its HPO and MONDO numbers are close to an
upper bound. The numbers to report come from real wording: the GSC+
evaluation of the HPO matching ([hpo-matching.md](hpo-matching.md)) and the
evaluation on ClinVar's condition names
([mondo-matching.md](mondo-matching.md)).

In [ ]:
bench.hpo_table(run)

## 4. Kinds of report

**Figure 7.** Report accuracy by kind of test, result and length, with 95%
intervals. Groups with few reports have wide intervals.

In [ ]:
bench.groups_figure(run, ("kind", "scenario", "variants", "pages"))

**Figure 8.** Report accuracy by challenge. Each challenge makes a report
harder in one known way.

In [ ]:
bench.groups_figure(run, ("tag",), "figure-challenges")

**Figure 9.** Report accuracy by template.

In [ ]:
bench.groups_figure(run, ("lab",), "figure-templates")

## 5. Checks for a person

Gentrex flags values that a person should look at. **Table 14** gives, for
each check, how many of the values that it looked at it flagged, how many
of the flagged ones are wrong (precision), and how many of the wrong ones
it flagged (recall).

In [ ]:
bench.checks_table(run)

**Table 15.** The same for whole reports. A report is flagged if a check
flagged one of its values, and wrong if it doesn't count as correct.

In [ ]:
bench.review_table(run)

## 6. Repeated runs

**Table 16.** Runs with the same setup, settings and code. The model server
batches requests differently each time, which changes a few answers.

In [ ]:
bench.repeats_table(runs)

## 7. Time and cost

**Table 17.** Seconds leave out the loading of the models. GPU-hours count
the GPUs of each command for as long as it ran.

In [ ]:
bench.efficiency_table(chosen)

## Appendix

**Table A1.** Every field of `RUN`. A missed value is in the report and not
in the record, an extra one is in the record and not in the report, and a
wrong one differs. `Recall given input` counts only the values that reached
`gentrex extract`. `Same text` is the share that is right character for
character.

In [ ]:
bench.fields_table(run)

**Table A2.** The clinical errors of the reports that count as wrong.

In [ ]:
bench.wrong_reports_table(run, errors)

**Table A3.** The first 50 values that aren't right. `errors` holds all of
them. Filter it to study one field, as in
`bench.errors_table(errors[errors["field"] == "mrn"])`.

In [ ]:
bench.errors_table(errors.head(50))